# 02 — f005 from saved artifacts

End-to-end **visualization-only** path for f005 single-unit PSTH categories.

This notebook:
- loads a saved A-family SPK epoch artifact via `jnwb.load_epoch_artifact`;
- uses an existing classification CSV when present;
- calls `run_f005_figure` (artifact-only; no NWB extraction).

**Build commands if artifacts are missing:**

```bash
python scripts/build_f005_afamily_spk_epochs.py --nwb-root <NWB_ROOT>
python scripts/classify_units_s_s_o.py \
  --epochs-p1 outputs/f005/afamily_spk_p1_epochs.npz \
  --unit-metadata outputs/f005/afamily_spk_p1_unit_metadata.csv
python figures/f005_unit_psth_categories.py
```

**Rule:** no local functions or classes.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd()
if not (REPO / "src").exists() and (REPO.parent / "src").exists():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import jnwb
from src.analysis.visualization.f005 import run_f005_figure

EPOCHS = REPO / "outputs" / "f005" / "afamily_spk_p1_epochs.npz"
CLASSIFICATION = REPO / "outputs" / "f005" / "classification" / "unit_classification.csv"
FIG_DIR = REPO / "figures" / "output"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Epoch artifact exists: {EPOCHS.exists()}")
print(f"Classification exists: {CLASSIFICATION.exists()}")

Epoch artifact exists: True
Classification exists: True


In [2]:
if EPOCHS.exists():
    batch = jnwb.load_epoch_artifact(EPOCHS)
    print(f"load_epoch_artifact shape={getattr(batch.data, 'shape', None)}")
    print(f"  manifest signal={batch.manifest.get('signal_class')}")
else:
    print("MISSING epoch artifact.")
    print("Build with: python scripts/build_f005_afamily_spk_epochs.py --nwb-root <NWB_ROOT>")

load_epoch_artifact shape=(60, 167, 5000)
  manifest signal=SPK


In [3]:
if not CLASSIFICATION.exists():
    print("MISSING classification CSV.")
    print(
        "Build with: python scripts/classify_units_s_s_o.py "
        "--epochs-p1 outputs/f005/afamily_spk_p1_epochs.npz "
        "--unit-metadata outputs/f005/afamily_spk_p1_unit_metadata.csv"
    )
else:
    import pandas as pd
    cls = pd.read_csv(CLASSIFICATION)
    print("display_class counts:")
    print(cls["display_class"].value_counts().to_string())

display_class counts:
display_class
unclassified    90
S-              44
S+              33


In [4]:
if EPOCHS.exists() and CLASSIFICATION.exists():
    manifest = run_f005_figure(
        EPOCHS,
        CLASSIFICATION,
        output_png=FIG_DIR / "f005_unit_psth_categories.png",
        output_svg=FIG_DIR / "f005_unit_psth_categories.svg",
        output_html=FIG_DIR / "f005_unit_psth_categories.html",
        manifest_path=FIG_DIR / "f005_unit_psth_categories_manifest.json",
        qa_csv_path=FIG_DIR / "f005_unit_psth_categories_qa.csv",
        allow_unknown_area=True,
    )
    print("run_f005_figure complete.")
    print(f"  class_counts={manifest.get('class_counts')}")
    print(f"  html={manifest.get('output_html')}")
else:
    print("SKIP: run_f005_figure requires epoch + classification artifacts.")
    print("CLI equivalent: python figures/f005_unit_psth_categories.py")

run_f005_figure complete.
  class_counts={'unclassified': 90, 'S-': 44, 'S+': 33}
  html=D:\workspace\omission\figures\output\f005_unit_psth_categories.html
